# Tarea Integradora 1 — Estadistica Descriptiva Aplicada a Series Financieras
### Plantilla de trabajo (borrador para completar en Colab)

**Curso:** Machine Learning / Deep Learning — Universidad del Norte
**Secciones integradas:** (1) Analisis y descripcion de datos · (2) Medidas de tendencia central, dispersion y dependencia lineal

---

## Antes de empezar

Este cuaderno no trae las respuestas resueltas. Es una plantilla guiada: contiene la estructura, las explicaciones teoricas necesarias, el codigo de descarga de datos ya resuelto (para no perder tiempo en eso) y espacios marcados con `# TODO` que cada estudiante debe completar por su cuenta.

Nota importante: la tarea evalua que el estudiante construya el razonamiento estadistico y financiero, no que el cuaderno "corra" sin que se entienda cada linea. Antes de copiar y pegar cualquier fragmento, léelo y verifica que entiendes qué hace.

### Como usar esta plantilla

1. Ejecuta las celdas en orden (de arriba hacia abajo), con `Shift + Enter`.
2. Cuando encuentres una celda con `# TODO`, reemplaza las lineas comentadas por tu propio codigo. No borres los comentarios guia: te sirven de mapa.
3. Cuando encuentres el texto "Respuesta (Markdown)", escribe tu interpretacion en texto, no solo el numero. La interpretacion financiera vale tanto como el calculo.
4. Si un error bloquea la ejecucion, lee el mensaje completo (la ultima linea suele indicar exactamente que falló) antes de pedir ayuda.
5. Guarda una copia en tu Drive (`Archivo -> Guardar una copia en Drive`) antes de empezar a editar.

### Minimo de Python necesario para quienes no tienen experiencia previa

| Necesito... | Lo hago con... | Ejemplo |
|---|---|---|
| Guardar un valor | `variable = valor` | `precio = 45.3` |
| Guardar una serie de datos | `pandas.Series` o `DataFrame` (ya se construye abajo) | `df["Close"]` |
| Calcular la media | `.mean()` | `df["retornos"].mean()` |
| Calcular la mediana | `.median()` | `df["retornos"].median()` |
| Calcular desviacion estandar (muestral) | `.std()` | `df["retornos"].std()` |
| Ver las primeras filas de una tabla | `.head()` | `df.head()` |
| Hacer un grafico | `matplotlib.pyplot` (`plt`) | `plt.plot(x, y)` |
| Filtrar filas de una tabla | `df[condicion]` | `df[df["retorno"] > 0]` |

No es necesario memorizar esta tabla: consúltala cuando la necesites. Tambien puedes ejecutar `help(funcion)` o preguntar a tu profesor como se aplica un metodo especifico, sin pedir que se resuelva el punto completo.


---
## Parte 0. Preparacion del entorno

Ejecuta la siguiente celda una sola vez al abrir el cuaderno. Instala y carga las librerias que se usaran durante toda la tarea.

No es necesario modificar nada en esta celda, pero lee los comentarios: usaras estas librerias de forma explicita en los puntos siguientes.


In [ ]:
# Instalacion de librerias necesarias (yfinance no viene preinstalada en Colab)
!pip install yfinance --quiet

# --- Librerias de manejo de datos ---
import pandas as pd            # Manejo de tablas de datos (DataFrames)
import numpy as np              # Calculo numerico (arreglos, funciones matematicas)

# --- Libreria de descarga de datos financieros ---
import yfinance as yf           # Descarga precios historicos desde Yahoo Finance

# --- Librerias de visualizacion ---
import matplotlib.pyplot as plt # Graficos generales (lineas, barras, histogramas, etc.)
import seaborn as sns           # Graficos estadisticos (se apoya en matplotlib)

# --- Libreria de estadistica ---
from scipy import stats         # Pruebas de normalidad (D'Agostino, Jarque-Bera), Q-Q plot, etc.

# Ajustes generales de estilo para los graficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Semilla para reproducibilidad en caso de usar algun muestreo aleatorio
np.random.seed(42)

print("Librerias cargadas correctamente.")


### 0.1 Descarga de los activos principales

Se trabajara con cuatro activos listados en Yahoo Finance:

| Simbolo | Emisor | Sector | Bolsa |
|---|---|---|---|
| `EC` | Ecopetrol S.A. (ADR) | Energia | NYSE |
| `CIB` | Bancolombia S.A. (ADR) | Financiero | NYSE |
| `TGLS` | Tecnoglass Inc. | Materiales / Industrial | NASDAQ |
| `^GSPC` | Indice S&P 500 | Referencia (benchmark) | Indice |

Periodo: los ultimos tres años de negociacion disponibles, frecuencia diaria. Las fechas deben quedar fijas en el codigo (no usar `datetime.now()`), para que el ejercicio sea reproducible por cualquier persona que lo ejecute despues.

La celda de abajo ya esta lista: solo ejecutala y observa su salida.


In [ ]:
# Lista de tickers a descargar
tickers = ["EC", "CIB", "TGLS", "^GSPC"]

# Fechas fijas (reproducibilidad). Ajusta si tu profesor indica otro periodo,
# pero una vez decidas las fechas, no las cambies durante el resto de la tarea.
fecha_inicio = "2022-01-01"
fecha_fin    = "2025-01-01"

# Descarga de precios (incluye Open, High, Low, Close, Adj Close, Volume)
data = yf.download(tickers, start=fecha_inicio, end=fecha_fin)

# Vista rapida de la estructura descargada
data.head()


In [ ]:
# Se extrae solo el precio de cierre ajustado y el volumen.
# El precio ajustado corrige splits y dividendos: es el estandar en analisis financiero.

precios = data["Close"]     # Si tu version de yfinance no trae "Adj Close" separado,
                              # revisa las columnas disponibles con: print(data.columns)
volumen = data["Volume"]

print("Precios de cierre (primeras filas):")
display(precios.head())

print("\nVolumen (primeras filas):")
display(volumen.head())


Importante: revisa con `data.columns` que columnas trajo realmente tu descarga (algunas versiones de `yfinance` cambian el nombre de "Adj Close"). Asegurate de trabajar con el precio de cierre ajustado, tal como lo exige el enunciado.

### 0.2 Calculo de rendimientos diarios

Para casi todos los puntos de la tarea se necesitan los rendimientos diarios (%), no los precios. El rendimiento simple del dia $t$ se calcula como:

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

En `pandas` existe un metodo que calcula esto directamente: `.pct_change()`.


In [ ]:
# TODO 0.2: calcula los rendimientos diarios (en proporcion, no en %) de "precios"
# Pista: pandas tiene un metodo que calcula el cambio porcentual entre filas consecutivas.
# rendimientos = precios.____()

rendimientos = None  # <-- reemplaza esta linea

# Una vez calculado, elimina la primera fila (quedara vacia porque no hay dia anterior)
# Pista: usa .dropna()
# rendimientos = rendimientos.____()

# Descomenta y ejecuta cuando tengas tu respuesta:
# display(rendimientos.head())


### 0.3 Guardado del dataset descargado

El enunciado exige anexar el conjunto de datos descargado en formato `.csv`. Completa la celda para guardar tanto los precios como los rendimientos.


In [ ]:
# TODO 0.3: guarda "precios" y "rendimientos" como archivos CSV independientes.
# Pista: el metodo de un DataFrame para exportar a CSV es .to_csv("nombre_archivo.csv")

# precios.____("precios_activos.csv")
# rendimientos.____("rendimientos_activos.csv")

print("Recuerda descargar ambos archivos .csv y anexarlos a tu entrega.")


---
## Parte I. Descripcion y clasificacion de datos (Puntos 1-11)

Recuerda de la Seccion 1 los conceptos de:
- Poblacion vs. muestra, y parametro vs. estadistico.
- Niveles de medicion: nominal, ordinal, discreta, continua.
- Distribuciones de frecuencias: $f_i$ (absoluta), $f_{ri}=f_i/N$ (relativa), $F_i$ (acumulada).
- Regla de Sturges para el numero de clases: $c = 1 + \dfrac{\ln(N)}{\ln(2)}$, y la amplitud $w = R/c$ (redondeada hacia arriba).
- Graficos apropiados segun el tipo de variable: barras, circular, Pareto, tallo y hojas, series temporales, histograma, ojiva.

### Punto 1 — Poblacion y muestra

Pregunta: identifique la poblacion y la muestra de este ejercicio y justifique su respuesta.


Respuesta (Markdown, no codigo):

- Poblacion: *(completa)*
- Muestra: *(completa)*
- Justificacion: *(completa, en 2-4 lineas, relacionando con la definicion de la Seccion 1)*


### Punto 2 — Clasificacion por nivel de medicion

Clasifique como nominal, ordinal, discreta o continua cada una de las siguientes variables, justificando cada respuesta con un criterio financiero (no basta con decir "es continua", explica por que):

1. Precio de cierre ajustado
2. Volumen diario
3. Sector economico del emisor
4. Signo del rendimiento diario (positivo/negativo/nulo)
5. Tipo de activo (accion / ADR / ETF)
6. Nivel de riesgo (Bajo/Medio/Alto)

Pista conceptual (Seccion 1): conteo => discreta; medicion => continua; categorias sin orden => nominal; categorias con orden pero sin distancia cuantificable => ordinal.


Respuesta (Markdown — tabla sugerida):

| Variable | Nivel de medicion | Justificacion |
|---|---|---|
| Precio de cierre ajustado | | |
| Volumen diario | | |
| Sector economico | | |
| Signo del rendimiento | | |
| Tipo de activo | | |
| Nivel de riesgo | | |


### Punto 3 — Tabla de frecuencias agrupada (TGLS)

Con los rendimientos diarios (%) de `TGLS`, construye una tabla de frecuencias agrupada usando la regla de Sturges. Debes reportar: intervalos, marca de clase ($X$), $f$, $f_r$, %, y $F$.

Recordatorio de formulas (Seccion 1):

$$R = U - L \qquad c = 1 + \frac{\ln(N)}{\ln(2)} \qquad w = \left\lceil \frac{R}{c} \right\rceil$$

Donde $U$ y $L$ son el maximo y minimo de la muestra, $c$ es el numero de clases (redondeado a un entero, usualmente entre 5 y 15) y $w$ es la amplitud de cada intervalo.


In [ ]:
# TODO 3: Tabla de frecuencias agrupada para los rendimientos (%) de TGLS

# Paso 1: extrae la serie de rendimientos de TGLS y expresala en porcentaje (multiplica por 100)
# rend_tgls = (rendimientos["TGLS"] * ____)

# Paso 2: calcula N (tamaño de la muestra), el minimo (L) y el maximo (U)
# N = len(____)
# L = rend_tgls.____()
# U = rend_tgls.____()
# R = U - L

# Paso 3: calcula el numero de clases con la regla de Sturges y redondea
# c = 1 + np.log(N) / np.log(2)
# c = round(c)   # redondea a entero

# Paso 4: calcula la amplitud w (redondea hacia arriba con np.ceil)
# w = np.ceil(R / c)

# Paso 5: construye los intervalos con pd.cut() o np.arange() + pd.cut()
# Pista: bins = np.arange(L, U + w, w)
# clases = pd.cut(rend_tgls, bins=bins, right=False)

# Paso 6: construye la tabla de frecuencias con .value_counts() y ordenala
# tabla_frecuencias = clases.value_counts().sort_index()

# Paso 7: calcula frecuencia relativa (f_r), porcentaje (%) y frecuencia acumulada (F)
# f_r = tabla_frecuencias / N
# porcentaje = f_r * 100
# F = tabla_frecuencias.cumsum()

# Paso 8: reune todo en un solo DataFrame y muestralo
# resumen = pd.DataFrame({"f": tabla_frecuencias, "f_r": f_r, "%": porcentaje, "F": F})
# display(resumen)

print("Completa los pasos anteriores y descomenta el codigo.")


### Punto 4 — Histograma y ojiva (TGLS)

Con la tabla del punto anterior, construye:
- Un histograma (barras verticales sobre los intervalos).
- Una ojiva (grafico de frecuencia acumulada, conectando puntos con lineas).

Recuerda (Seccion 1): el histograma es la primera herramienta exploratoria de la forma de la distribucion (simetria/sesgo); la ojiva responde preguntas tipo "que % de dias tuvo un rendimiento menor a X%?".


In [ ]:
# TODO 4a: Histograma de los rendimientos de TGLS
# Pista: plt.hist(datos, bins=numero_de_clases, edgecolor="black")

fig, ax = plt.subplots()
# ax.hist(____, bins=____, edgecolor="black", color="steelblue")
ax.set_title("Histogram of TGLS Daily Returns")   # Titulos en ingles (requisito)
ax.set_xlabel("Daily Return (%)")
ax.set_ylabel("Frequency")
plt.show()


In [ ]:
# TODO 4b: Ojiva (frecuencia acumulada) de los rendimientos de TGLS
# Pista: usa los limites superiores de cada intervalo (eje x) y la columna F de tu tabla (eje y)
# plt.plot(limites_superiores, F, marker="o")

fig, ax = plt.subplots()
# ax.plot(____, ____, marker="o", color="darkorange")
ax.set_title("Ogive of TGLS Daily Returns")
ax.set_xlabel("Daily Return (%) - upper class limit")
ax.set_ylabel("Cumulative Frequency")
plt.show()


### Punto 5 — Series temporales de precios (los 4 activos)

Elabora un grafico de series temporales del precio de cierre ajustado de cada uno de los cuatro activos.

Pista: si los cuatro activos tienen escalas muy distintas (por ejemplo, `^GSPC` esta en miles y `EC` en decenas), puede ser mas claro usar subplots (uno por activo) en vez de una sola grafica superpuesta. Decide y justifica tu elección.


In [ ]:
# TODO 5: Grafico de series temporales de los precios de cierre ajustados

fig, ax = plt.subplots()
# for activo in tickers:
#     ax.plot(precios.index, precios[activo], label=activo)
ax.set_title("Adjusted Closing Price Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend()
plt.show()

# Si los precios tienen escalas muy distintas, considera usar subplots:
# fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 8))
# ... (uno por activo)


### Punto 6 — Barras de rendimiento promedio anual y participacion en portafolio

1. Diagrama de barras con el rendimiento promedio anual de cada activo, por año (un grupo de barras por año, una barra por activo, o viceversa).
2. Diagrama circular con la participacion de cada activo en un portafolio de pesos iguales (con 4 activos, cada uno pesa 25%).

Pista para (1): agrupa los rendimientos diarios por año. `pandas` permite hacerlo con `.resample("Y")`.


In [ ]:
# TODO 6a: Rendimiento promedio anual por activo (agrupado por año)
# Pista: rendimientos.resample("Y").mean() agrupa por año y calcula el promedio de cada columna

# rend_anual = rendimientos.resample(____).____()
# display(rend_anual)

# Luego grafica un diagrama de barras (agrupado o por facetas)
fig, ax = plt.subplots()
# rend_anual.plot(kind="bar", ax=ax)
ax.set_title("Average Annual Return by Asset")
ax.set_xlabel("Year")
ax.set_ylabel("Average Daily Return (%)")
plt.show()


In [ ]:
# TODO 6b: Diagrama circular de un portafolio de pesos iguales
# Con 4 activos, cada peso es 1/4 = 25%. Construyelo como una lista o Serie.

# pesos = pd.Series([0.25, 0.25, 0.25, 0.25], index=tickers)

fig, ax = plt.subplots()
# ax.pie(pesos, labels=pesos.index, autopct="%1.1f%%")
ax.set_title("Equal-Weighted Portfolio Composition")
plt.show()


### Punto 7 — Tallo y hojas (EC, un mes cualquiera)

Con los rendimientos diarios de `EC` de un mes cualquiera (a elección propia), construye un diagrama de tallo y hojas. Reporta el maximo, el minimo y el rango.

Pista: Python no tiene una funcion nativa de una sola linea para tallo y hojas, pero la libreria `stemgraphic` si. Tambien puedes construirlo manualmente (mas trabajo, pero mas control), como se explico en la Seccion 1: tallo = parte entera, hoja = primer decimal.


In [ ]:
# Si decides usar la libreria especializada, descomenta e instala:
# !pip install stemgraphic --quiet
# import stemgraphic

# TODO 7: selecciona un mes cualquiera de EC (ej. enero de 2023) y filtra los rendimientos (%)
# Pista: puedes filtrar por fecha asi -> rendimientos.loc["2023-01":"2023-01", "EC"] * 100

# rend_ec_mes = ____

# Construye el diagrama de tallo y hojas (con stemgraphic.stem_graphic(rend_ec_mes)
# o manualmente clasificando cada valor en su tallo)

# Reporta maximo, minimo y rango:
# maximo = rend_ec_mes.max()
# minimo = rend_ec_mes.min()
# rango  = maximo - minimo
# print(f"Maximo: {maximo}, Minimo: {minimo}, Rango: {rango}")


### Punto 8 — Pareto del signo del rendimiento (CIB)

1. Clasifica cada dia de `CIB` segun el signo del rendimiento (positivo / negativo / nulo).
2. Construye un diagrama de Pareto con la frecuencia de cada categoria (barras ordenadas de mayor a menor, mas una linea de porcentaje acumulado).

Pista: usa `np.sign()` sobre los rendimientos y luego mapea los valores {-1, 0, 1} a etiquetas de texto con `.map({...})`.


In [ ]:
# TODO 8a: clasifica el signo del rendimiento de CIB
# signo = np.sign(rendimientos["CIB"])
# etiquetas_signo = signo.map({1: "Positivo", -1: "Negativo", 0: "Nulo"})

# conteo_signo = etiquetas_signo.value_counts()  # ya viene ordenado de mayor a menor
# display(conteo_signo)


In [ ]:
# TODO 8b: Diagrama de Pareto (barras + linea de % acumulado)
# Pista: calcula el porcentaje acumulado con .cumsum() / .sum() * 100

# porcentaje_acumulado = conteo_signo.cumsum() / conteo_signo.sum() * 100

fig, ax1 = plt.subplots()
# ax1.bar(conteo_signo.index, conteo_signo.values, color="steelblue")
ax1.set_xlabel("Return Sign")
ax1.set_ylabel("Frequency")
ax1.set_title("Pareto Chart - CIB Daily Return Sign")

ax2 = ax1.twinx()  # segundo eje y, para el % acumulado
# ax2.plot(conteo_signo.index, porcentaje_acumulado.values, color="red", marker="o")
ax2.set_ylabel("Cumulative Percentage")
ax2.set_ylim(0, 110)
plt.show()


### Punto 9 — Variable nominal: sector economico (universo ampliado)

Antes de este punto, construye el universo ampliado: ademas de `EC`, `CIB` y `TGLS`, selecciona minimo 10 activos adicionales que cubran al menos 4 sectores economicos distintos. Verifica el sector de cada uno con `yfinance` (`ticker.info["sector"]`).


In [ ]:
# TODO (previo al Punto 9): construccion del universo ampliado
# Ejemplo de estructura sugerida: completa con tus propios tickers

# universo_ampliado = ["EC", "CIB", "TGLS", "____", "____", "____", "____",
#                       "____", "____", "____", "____", "____", "____"]

# Para consultar el sector de cada ticker:
# info_activos = {}
# for tk in universo_ampliado:
#     activo = yf.Ticker(tk)
#     info_activos[tk] = activo.info.get("sector", "Desconocido")

# tabla_sectores = pd.Series(info_activos, name="Sector")
# display(tabla_sectores)


Con ese universo, construye la distribucion de frecuencias de la variable nominal sector economico (categoria, $f$, $f_r$, %) y elabora un diagrama de barras y un diagrama circular.

Preguntas de interpretacion (respondelas en Markdown):
- Tiene sentido calcular una frecuencia acumulada ($F$) para esta variable? Justifica.
- Que medida de tendencia central es la unica estadisticamente valida para resumirla? Calculala e interpretala.


In [ ]:
# TODO 9: distribucion de frecuencias de "sector"
# frecuencia_sector = tabla_sectores.value_counts()
# f_r_sector = frecuencia_sector / frecuencia_sector.sum()
# porcentaje_sector = f_r_sector * 100

# resumen_sector = pd.DataFrame({"f": frecuencia_sector, "f_r": f_r_sector, "%": porcentaje_sector})
# display(resumen_sector)

# Diagrama de barras
fig, ax = plt.subplots()
# frecuencia_sector.plot(kind="bar", ax=ax, color="seagreen")
ax.set_title("Frequency Distribution by Economic Sector")
ax.set_xlabel("Sector")
ax.set_ylabel("Frequency")
plt.show()

# Diagrama circular
fig, ax = plt.subplots()
# ax.pie(frecuencia_sector, labels=frecuencia_sector.index, autopct="%1.1f%%")
ax.set_title("Sector Distribution - Extended Universe")
plt.show()


Respuestas de interpretacion (Punto 9):

- F acumulada tiene sentido aqui? *(completa)*
- Medida de tendencia central valida y su valor: *(completa)*


### Punto 10 — Variable ordinal: nivel de riesgo

Con los rendimientos diarios del universo ampliado, construye la variable ordinal nivel de riesgo (`Bajo`, `Medio`, `Alto`), clasificando cada activo segun su desviacion estandar anualizada dividida en terciles.

Recuerda (Seccion 2): $\sigma_{\text{anual}} = \sigma_{\text{diaria}} \times \sqrt{252}$.

Sugerencia tecnica del enunciado:
```python
riesgo = pd.Categorical(
    etiquetas,
    categories=["Bajo","Medio","Alto"],
    ordered=True)
```


In [ ]:
# TODO 10a: calcula la desviacion estandar diaria de cada activo del universo ampliado
# y anualizala con la regla de la raiz del tiempo

# rendimientos_ampliado = ____   # descarga o reutiliza los rendimientos del universo ampliado
# std_diaria = rendimientos_ampliado.____()
# std_anual = std_diaria * np.sqrt(252)
# display(std_anual.sort_values())


In [ ]:
# TODO 10b: clasifica en terciles (Bajo/Medio/Alto) usando pd.qcut
# Pista: pd.qcut(datos, q=3, labels=["Bajo", "Medio", "Alto"])

# etiquetas_riesgo = pd.qcut(std_anual, q=3, labels=["Bajo", "Medio", "Alto"])

# Convertela en Categorical ordenado (tal como indica el enunciado)
# riesgo = pd.Categorical(etiquetas_riesgo, categories=["Bajo", "Medio", "Alto"], ordered=True)

# display(pd.Series(riesgo, index=std_anual.index, name="Nivel de riesgo"))


Con esta variable, construye su distribucion de frecuencias (incluyendo $F$, respetando el orden `Bajo < Medio < Alto`) y grafícala con un diagrama de barras ordenado (no circular).

Preguntas de interpretacion:
- Por que la $F$ acumulada si es interpretable aqui, a diferencia del punto 9?
- Por que un diagrama circular no es apropiado para esta variable?
- Reporta la moda y la mediana del nivel de riesgo. Explica por que no es valido calcular la media aritmetica sobre `Bajo=1, Medio=2, Alto=3`.


In [ ]:
# TODO 10c: distribucion de frecuencias ordenada + grafico de barras ordenado

# frecuencia_riesgo = pd.Series(riesgo).value_counts().reindex(["Bajo", "Medio", "Alto"])
# F_riesgo = frecuencia_riesgo.cumsum()
# display(pd.DataFrame({"f": frecuencia_riesgo, "F": F_riesgo}))

fig, ax = plt.subplots()
# frecuencia_riesgo.plot(kind="bar", ax=ax, color=["seagreen", "goldenrod", "firebrick"])
ax.set_title("Ordered Distribution of Risk Level")
ax.set_xlabel("Risk Level")
ax.set_ylabel("Frequency")
plt.show()


Respuestas de interpretacion (Punto 10):

- Por que F si tiene sentido aqui? *(completa)*
- Por que no un diagrama circular? *(completa)*
- Moda: *(completa)* — Mediana: *(completa)*
- Por que no es valida la media aritmetica sobre 1,2,3? *(completa)*


### Punto 11 — Sintesis: escalas de medicion y estadisticos validos

Completa la siguiente tabla para cada variable de los puntos 2, 9 y 10.


Tabla de sintesis (Markdown):

| Variable | Nivel de medicion | Medidas de tendencia central validas | Medidas de dispersion validas | Grafico(s) apropiado(s) |
|---|---|---|---|---|
| Precio de cierre ajustado | | | | |
| Volumen diario | | | | |
| Sector economico | | | | |
| Signo del rendimiento | | | | |
| Tipo de activo | | | | |
| Nivel de riesgo | | | | |


---
## Parte II. Tendencia central, dispersion, normalidad y dependencia lineal (Puntos 12-19)

Recuerda de la Seccion 2:

$$\bar{x}=\frac{\sum x_i}{n}\qquad \tilde x = \text{valor central} \qquad G=\Big[\prod(1+r_i)\Big]^{1/n}-1$$
$$s^2=\frac{\sum(x_i-\bar x)^2}{n-1}\qquad s=\sqrt{s^2}\qquad \text{DM}=\frac{\sum|x_i-\bar x|}{n}\qquad \text{CV}=\frac{s}{\bar x}\times 100\%$$
$$s_{xy}=\frac{\sum(x_i-\bar x)(y_i-\bar y)}{n-1}\qquad r=\frac{s_{xy}}{s_x s_y}$$

### Punto 12 — Media, mediana y moda de los 4 activos principales

Calcula media, mediana y moda de los rendimientos diarios de `EC`, `CIB`, `TGLS` y `^GSPC`. Compara los tres valores por activo y explica que indica su relacion sobre la forma de la distribucion (recuerda: si moda < mediana < media, hay asimetria positiva, y viceversa).

Nota tecnica: los rendimientos diarios rara vez tienen una moda "clasica" (valores repetidos exactos son poco comunes en datos continuos). Puedes usar `scipy.stats.mode()`, o discutir por que la moda es poco informativa para variables continuas; ambas rutas son validas si las justificas.


In [ ]:
# TODO 12: media, mediana y moda de los 4 activos

# media_activos = rendimientos.____()
# mediana_activos = rendimientos.____()
# moda_activos = rendimientos.____()   # considera usar .mode() de pandas o discutir su limitacion

# resumen_tendencia = pd.DataFrame({
#     "Media": media_activos,
#     "Mediana": mediana_activos,
#     # "Moda": moda_activos,
# })
# display(resumen_tendencia)


### Punto 13 — Media geometrica y CAGR

Calcula la media geometrica de cada activo y el CAGR del periodo completo. Explica por que la media aritmetica no es valida para resumir rendimientos compuestos (recuerda el ejemplo de la "fuga de volatilidad" de la Seccion 2).

Formulas:
$$\bar r_G = \Big[\prod_{i=1}^n (1+r_i)\Big]^{1/n} - 1 \qquad \text{CAGR} = \left(\frac{P_{\text{final}}}{P_{\text{inicial}}}\right)^{1/\text{años}} - 1$$


In [ ]:
# TODO 13a: media geometrica de los rendimientos diarios de cada activo
# Pista: G = (1 + rendimientos).prod() ** (1/n) - 1   (aplicalo columna por columna)

# n = len(rendimientos)
# media_geometrica = (1 + rendimientos).____().____(1/n) - 1
# display(media_geometrica)


In [ ]:
# TODO 13b: CAGR del periodo completo, usando el precio inicial y final de cada activo
# Pista: calcula el numero de años del periodo (aprox. dias_totales / 252, o usa fechas exactas)

# precio_inicial = precios.iloc[0]
# precio_final = precios.iloc[-1]
# n_anios = ____  # calcula el numero de años entre fecha_inicio y fecha_fin

# cagr = (precio_final / precio_inicial) ** (1 / n_anios) - 1
# display(cagr)


Explicacion (Markdown): por que la media aritmetica sobrestima el rendimiento real cuando hay compuestos?

*(completa)*


### Punto 14 — Rango, varianza, desviacion estandar, DM y CV

Calcula, para cada activo: rango ($R=U-L$), varianza ($s^2$), desviacion estandar ($s$), desviacion media (DM) y coeficiente de variacion (CV). Ordena los activos de mayor a menor riesgo segun cada medida (pueden no coincidir).

Advertencia (Seccion 2): el CV solo es interpretable cuando la media esta lejos de cero. Los rendimientos diarios suelen tener medias cercanas a cero: comenta si el CV es confiable en este caso.


In [ ]:
# TODO 14: tabla resumen de dispersion para los 4 activos

# rango = rendimientos.____() - rendimientos.____()
# varianza = rendimientos.____()          # pandas usa n-1 por defecto (muestral); confirma esto
# desv_estandar = rendimientos.____()
# desv_media = (rendimientos - rendimientos.mean()).____().____()   # valor absoluto y luego media
# cv = desv_estandar / rendimientos.mean() * 100

# resumen_dispersion = pd.DataFrame({
#     "Rango": rango,
#     "Varianza": varianza,
#     "Desv. Estandar": desv_estandar,
#     "DM": desv_media,
#     "CV (%)": cv
# })
# display(resumen_dispersion)

# Ordena por cada columna (ejemplo con desviacion estandar):
# display(resumen_dispersion.sort_values("Desv. Estandar", ascending=False))


### Punto 15 — Simetria y normalidad (asimetria, D'Agostino, Jarque-Bera)

Para cada activo, evalua la simetria con: el coeficiente de asimetria, la prueba de D'Agostino (`scipy.stats.skewtest`) y la prueba de Jarque-Bera (`scipy.stats.jarque_bera`). Reporta estadistico y valor $p$; concluye al 5% de significancia.

Recuerda las hipotesis:
$$H_0: \text{Sk} = 0 \text{ (simetrica)} \qquad H_1: \text{Sk} \neq 0 \text{ (asimetrica)}$$


In [ ]:
# TODO 15: pruebas de simetria/normalidad para cada activo

resultados_normalidad = {}
for activo in tickers:
    serie = rendimientos[activo].dropna()

    # asimetria = serie.____()   # pista: .skew()
    # dagostino_stat, dagostino_p = stats.skewtest(serie)
    # jb_stat, jb_p = stats.jarque_bera(serie)

    # resultados_normalidad[activo] = {
    #     "Asimetria": asimetria,
    #     "D'Agostino stat": dagostino_stat, "D'Agostino p-valor": dagostino_p,
    #     "Jarque-Bera stat": jb_stat, "Jarque-Bera p-valor": jb_p,
    # }
    pass

# tabla_normalidad = pd.DataFrame(resultados_normalidad).T
# display(tabla_normalidad)


Conclusion al 5% de significancia (Markdown): para cada activo, se rechaza o no la hipotesis de simetria/normalidad? *(completa)*


### Punto 16 — Q-Q plot de CIB

Construye el Q-Q plot de los rendimientos de `CIB` frente a la normal teorica. Interpreta el ajuste a la recta, especialmente en las colas.

Pista: `scipy.stats.probplot(datos, dist="norm", plot=plt)` genera el Q-Q plot en una sola linea.


In [ ]:
# TODO 16: Q-Q plot de CIB

fig, ax = plt.subplots()
# stats.probplot(rendimientos["CIB"].dropna(), dist="norm", plot=ax)
ax.set_title("Q-Q Plot - CIB Daily Returns vs. Normal Distribution")
plt.show()


Interpretacion (Markdown): los puntos se alinean con la recta? que ocurre en las colas? *(completa)*


### Punto 17 — Matrices de covarianza y correlacion

Calcula las matrices de covarianza y correlacion entre `EC`, `CIB` y `TGLS`. Identifica el par con mayor y con menor potencial de diversificacion.

Pista: `DataFrame.cov()` y `DataFrame.corr()` calculan estas matrices directamente sobre las columnas.


In [ ]:
# TODO 17: matrices de covarianza y correlacion (solo EC, CIB, TGLS)

# activos_3 = rendimientos[["EC", "CIB", "TGLS"]]
# matriz_cov = activos_3.____()
# matriz_corr = activos_3.____()

# display(matriz_cov)
# display(matriz_corr)

# Sugerencia: visualizalas con un heatmap
# sns.heatmap(matriz_corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
# plt.title("Correlation Matrix - EC, CIB, TGLS")
# plt.show()


Respuesta (Markdown): que par de activos tiene mayor potencial de diversificacion (correlacion mas baja o negativa)? cual tiene menor? *(completa)*


### Punto 18 — Varianza de un portafolio de dos activos

Con el par de mayor diversificacion potencial, construye un portafolio con $w_1=w_2=0.5$. Calcula $\sigma_P^2$ y $\sigma_P$ y compáralas con el promedio ponderado de las varianzas individuales. Repite con el par mas correlacionado positivamente.

Formula (Seccion 2):
$$\sigma_P^2 = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2 w_1 w_2 \sigma_{12}$$


In [ ]:
# TODO 18: varianza del portafolio para ambos pares de activos

def varianza_portafolio(sigma1_2, sigma2_2, sigma12, w1=0.5, w2=0.5):
    # TODO: implementa la formula de sigma_P^2
    # return w1**2 * sigma1_2 + w2**2 * sigma2_2 + 2 * w1 * w2 * sigma12
    pass

# Par de mayor diversificacion (reemplaza "A" y "B" por los tickers correspondientes):
# var_A = rendimientos["A"].var()
# var_B = rendimientos["B"].var()
# cov_AB = matriz_cov.loc["A", "B"]
# sigma_p2_diversificado = varianza_portafolio(var_A, var_B, cov_AB)
# sigma_p_diversificado = np.sqrt(sigma_p2_diversificado)

# Compara con el promedio ponderado simple:
# promedio_ponderado = 0.5 * var_A + 0.5 * var_B

# print(f"Varianza portafolio diversificado: {sigma_p2_diversificado}")
# print(f"Promedio ponderado de varianzas: {promedio_ponderado}")

# Repite el mismo procedimiento para el par mas correlacionado positivamente


### Punto 19 — Conclusion integradora

Redacta una conclusion integradora (maximo una pagina, en Markdown) que relacione:
- Los hallazgos descriptivos, incluyendo las variables nominal y ordinal (puntos 9-11).
- Las medidas de tendencia central, dispersion y dependencia lineal (puntos 12-18).
- Una recomendacion de inversion para un inversionista con baja tolerancia al riesgo.


Conclusion integradora (Markdown, maximo 1 pagina):

*(completa)*

---


## Checklist final antes de entregar

- Todas las celdas ejecutan de arriba hacia abajo sin errores (prueba con `Entorno de ejecucion -> Reiniciar y ejecutar todo`).
- Cada resultado numerico tiene su interpretacion financiera en una celda de Markdown.
- Los titulos, ejes y anotaciones de las figuras estan en ingles.
- Las variables nominal y ordinal estan codificadas explicitamente con `dtype="category"` / `pd.Categorical(..., ordered=True)`.
- Se anexan los archivos `.csv` de los datos descargados (activos principales y universo ampliado).
- El cuaderno esta compartido con permisos de visualizacion y edicion para `lihkir@uninorte.edu.co`.
- El codigo tiene comentarios claros, no solo los que trae esta plantilla.

### Donde buscar ayuda

- Documentacion oficial de pandas: https://pandas.pydata.org/docs/
- Documentacion oficial de yfinance: https://pypi.org/project/yfinance/
- Documentacion de scipy.stats: https://docs.scipy.org/doc/scipy/reference/stats.html
- Diapositivas de las Secciones 1 y 2 del curso: cada formula usada aqui aparece explicada y con ejemplos.
